Pre process

In [1]:
import pandas as pd
import numpy as np
import math
import torch

In [2]:
df=pd.read_csv("./datasets/cleaned_data.csv")
df.drop(columns=["sn"], inplace=True)
df["month"]=df["month"].map({
    1:"january",
    2:"february",
    3:"march",
    4:"april",
    5:"may",
    6:"june",
    7:"july",
    8:"august",
    9:"september",
    10:"october",
    11:"november",
    12:"december"
})

In [3]:
month_order = "january,february,march,april,may,june,july,august,september,october,november,december".split(",")
df["month"] = pd.Categorical(df["month"], categories=month_order, ordered=True)

item_order = [i+1 for i in range(100)]
df["item_id"] = pd.Categorical(df["item_id"], categories=item_order, ordered=True)

month = pd.get_dummies(df["month"], prefix="month", dtype=int)
item_id = pd.get_dummies(df["item_id"], prefix="item_id", dtype=int)

df.drop(columns=["month", "item_id", "date", "name", "year", "kcal", "price"], inplace=True)
df = pd.concat([df, item_id, month], axis=1)

df["item_count"] = df["item_count"].apply(np.log1p)

day_min = 1
day_max = 32
df["day"] = (df["day"] - day_min) / (day_max - day_min)  # simple min-max normalization

target_min = df["item_count"].min()
target_max = df["item_count"].max()
df["item_count"] = (df["item_count"] - target_min) / (target_max - target_min)

df = df[df["item_count"] > 0].copy()

df = df.sample(frac=1, random_state=42)

df.head()

,item_count,day,item_id_1,item_id_2,item_id_3,item_id_4,item_id_5,item_id_6,item_id_7,item_id_8,...,month_march,month_april,month_may,month_june,month_july,month_august,month_september,month_october,month_november,month_december
6603,0.455364,0.225806,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
35600,0.218404,0.709677,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
100417,0.327606,0.000000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
41212,0.218404,0.516129,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97809,0.696167,0.129032,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


split train and test

In [4]:
def split_train_test(x,y,size):
    train_len=math.floor(len(y)*size)
    
    x_train=torch.tensor(x[0:train_len], dtype=torch.float32, device="cuda", requires_grad=True)
    y_train=torch.tensor(y[0:train_len], dtype=torch.float32, device="cuda")
    
    x_test=torch.tensor(x[train_len:], dtype=torch.float32, device="cuda", requires_grad=True)
    y_test=torch.tensor(y[train_len:], dtype=torch.float32, device="cuda")
    
    return x_train,y_train,x_test,y_test

In [5]:
x_train,y_train,x_test,y_test = split_train_test(x=df.drop(columns=["item_count"]).values, y=(df["item_count"].values), size=0.8)

In [15]:
x_test.shape

torch.Size([4697, 113])

In [6]:
class Model :
    
    def __init__(self, x, y, neurons=[8,8], device="cuda"):
        self.x = x.to(device)
        self.y = y.view(-1,1).to(device)
        self.device=device
        #shape of x
        N,d = x.shape
        #weight and bias of first
        self.w1 = (torch.randn(d, neurons[0], dtype=torch.float32, device=device)*(2/d)**0.5).requires_grad_()
        self.b1 = torch.randn(neurons[0], dtype=torch.float32, device=device, requires_grad=True)
        #weight and bias of second neurns
        self.w2 = (torch.randn(neurons[0], neurons[1], dtype=torch.float32, device=device)*(2/neurons[0])**0.5).requires_grad_()
        self.b2 = torch.randn(neurons[1], dtype=torch.float32, device=device, requires_grad=True)
        #weight and bias of third neurons
        self.w3 = (torch.randn(neurons[1], 1, dtype=torch.float32, device=device)*(2/neurons[1])**0.5).requires_grad_()
        self.b3 = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)
    def train(self, epochs=1000, batch_size=1, lr=0.01):
        
        for epoch in range(epochs):
            curr_batch=0
            row_len = self.x.shape[0]
            
            total_loss = 0
            
            while(curr_batch < row_len):
                
                curr_batch_len=batch_size
                if row_len-curr_batch >= batch_size : 
                    curr_batch_len=batch_size
                else:
                    curr_batch_len = row_len-curr_batch
                
                x_batch = self.x[curr_batch:curr_batch+curr_batch_len]
                y_batch = self.y[curr_batch:curr_batch+curr_batch_len]
                
                n1 = torch.relu( x_batch @ self.w1 + self.b1)
                n2 = torch.relu(n1 @ self.w2 + self.b2)
                n3 = n2 @ self.w3 + self.b3
                
                error = n3 - y_batch
                
                mse = (error**2).mean()
                
                total_loss += mse.item() * curr_batch_len
                
                mse.backward()
                
                with torch.no_grad():
                    self.w1 -= lr*self.w1.grad
                    self.b1 -= lr*self.b1.grad
                    self.w2 -= lr*self.w2.grad
                    self.b2 -= lr*self.b2.grad
                    self.w3 -= lr*self.w3.grad
                    self.b3 -= lr*self.b3.grad
                
                self.w1.grad.zero_()
                self.b1.grad.zero_()
                self.w2.grad.zero_()
                self.b2.grad.zero_()
                self.w3.grad.zero_()
                self.b3.grad.zero_()
                
                curr_batch+=curr_batch_len
            
            if (epoch+1)%(1000)==0:
                print(f"Epoch====({epoch+1}): Loss={(total_loss/row_len)}")
    
    def predict(self, x_new):
        x_new = x_new.to(self.device)
        n1 = torch.relu(x_new @ self.w1 + self.b1)
        n2 = torch.relu(n1 @ self.w2 + self.b2 )
        n3 = n2 @ self.w3 + self.b3 
        return n3

In [7]:
model = Model(x_train, y_train, neurons=[256, 128], device="cuda")

model.train(epochs=70000, batch_size=50000, lr=0.0003)

Epoch====(1000): Loss=0.059445612132549286
Epoch====(2000): Loss=0.05290992185473442
Epoch====(3000): Loss=0.0478459969162941
Epoch====(4000): Loss=0.04367617145180702
Epoch====(5000): Loss=0.04015440121293068
Epoch====(6000): Loss=0.037071578204631805
Epoch====(7000): Loss=0.034199900925159454
Epoch====(8000): Loss=0.031867045909166336
Epoch====(9000): Loss=0.02980993315577507
Epoch====(10000): Loss=0.027890589088201523
Epoch====(11000): Loss=0.026117419824004173
Epoch====(12000): Loss=0.024568062275648117
Epoch====(13000): Loss=0.02319919504225254
Epoch====(14000): Loss=0.021951381117105484
Epoch====(15000): Loss=0.020831042900681496
Epoch====(16000): Loss=0.0198208075016737
Epoch====(17000): Loss=0.018895301967859268
Epoch====(18000): Loss=0.018050605431199074
Epoch====(19000): Loss=0.017276674509048462
Epoch====(20000): Loss=0.01656517945230007
Epoch====(21000): Loss=0.015910211950540543
Epoch====(22000): Loss=0.015305974520742893
Epoch====(23000): Loss=0.014748270623385906
Epoch==

In [8]:
y_pred = model.predict(x_test)

y_log = y_pred * (target_max - target_min) + target_min
y_pred_real = torch.expm1(y_log)
torch.set_printoptions(sci_mode=False)
y_pred_real[0:100]

tensor([[  9.5736],
        [ 70.1656],
        [  0.5996],
        [ 50.3610],
        [  1.3384],
        [ 24.5482],
        [  3.8955],
        [229.6151],
        [  1.8225],
        [  2.5323],
        [  0.2911],
        [ 26.0682],
        [  7.3693],
        [  6.7512],
        [  0.6774],
        [118.6512],
        [  0.4433],
        [ 16.7378],
        [ 22.4982],
        [ 13.0707],
        [ 21.7346],
        [  1.7063],
        [ 30.1326],
        [ 38.2790],
        [ 22.7558],
        [ 21.7614],
        [228.7331],
        [  3.4473],
        [ 25.8698],
        [  3.7027],
        [ 69.5866],
        [  0.9123],
        [  4.2918],
        [  2.4916],
        [  4.6901],
        [  4.1932],
        [  4.6223],
        [  6.5822],
        [  7.7956],
        [  2.8665],
        [  2.4529],
        [ 65.9605],
        [133.0423],
        [  2.0889],
        [246.4053],
        [  8.6387],
        [  2.4475],
        [  5.0370],
        [ 25.0913],
        [  2.3498],


In [9]:
def r2_score(y_true, y_pred):
    # make sure y_true and y_pred are the same shape
    y_true = y_true.view(-1, 1)
    y_pred = y_pred.view(-1, 1)
    
    # total sum of squares
    ss_tot = ((y_true - y_true.mean())**2).sum()
    
    # residual sum of squares
    ss_res = ((y_true - y_pred)**2).sum()
    
    r2 = 1 - ss_res / ss_tot
    return r2.item()

In [10]:
accuracy = r2_score(y_test, y_pred)
print(f"R² = {accuracy:.4f}")  

R² = 0.8776


Export data

In [11]:
import torch.nn as nn

class ONNXWrapper(nn.Module):
    def __init__(self, manual_model, target_min, target_max):
        super().__init__()
        self.model = manual_model
        
        # store as tensors so ONNX keeps them
        self.target_min = torch.tensor(target_min, dtype=torch.float32)
        self.target_max = torch.tensor(target_max, dtype=torch.float32)

    def forward(self, x):
        # 1️⃣ Get normalized prediction
        y_norm = self.model.predict(x)

        # 2️⃣ Reverse min-max scaling
        y_log = y_norm * (self.target_max - self.target_min) + self.target_min

        # 3️⃣ Reverse log1p
        y_original = torch.expm1(y_log)

        return y_original

onnx_model = ONNXWrapper(model, target_min, target_max)
onnx_model.eval()

dummy_input = torch.randn(1, x_train.shape[1], device="cpu")

torch.onnx.export(
    onnx_model,
    dummy_input,
    "sales_prediction.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=18
)

print("Single-file ONNX model exported successfully!")

C:\Users\tikad\AppData\Local\Temp\ipykernel_2804\736185919.py:29: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ONNXWrapper()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ONNXWrapper()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_0` requires gradient. but it's not properly registered as a parameter. torch.export will detach it and treat it as a constant tensor but please register it as parameter instead.
  unlift_gm = _create_stateful_graph_module(new_gm, ep.range_constraints, ep)
c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_1` requires gradient. but it's not properly registered as a parameter. torch.export will detach it and treat it as a constant tensor but please register it as parameter instead.
  unlift_gm = _create_stateful_graph_module(new_gm, ep.range_constraints, ep)
c:\Projects\machine_learning\demand_forecasting\model_training\env\Lib\site-packages\torch\export\_unlift.py:825: UserWarning: A model attribute `lifted_tensor_2` requi

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
Single-file ONNX model exported successfully!


C:\Users\tikad\AppData\Local\Programs\Python\Python312\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
